# FRIDA-Decisions quickstart

Structured decisions over text — **choice**, **score**, **yes/no** and **ranking** — from one
encoder pass. This notebook installs the package, loads the model from the Hugging Face Hub,
runs a first request, shows each question type, scores a 243-intent catalog, and runs the
int8 ONNX model on CPU.

On Colab, a GPU runtime (Runtime → Change runtime type → T4 GPU) makes the torch cells fast;
the ONNX cell runs on CPU either way.

In [ ]:
!pip install -q "frida-decisions[onnx] @ git+https://github.com/ai-forever/frida-decisions"

In [ ]:
import json, os, time
import torch
from frida_decisions import Judge, OnnxJudge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
judge = Judge.from_pretrained(MODEL, device=device)
print("loaded on", device)

## First request

A request is a `state` (the text being judged) and named `questions`. All questions are
answered from the same encoder pass.

In [ ]:
request = {
    "state": "Добрый день. Третий день не могу войти в личный кабинет: пишет, что пароль неверный, "
             "а письмо для сброса не приходит. Из-за этого не могу скачать счёт, срок оплаты завтра.",
    "questions": {
        "team": {"type": "choice", "instructions": "В какую команду направить обращение?",
                 "criteria": {"auth": "доступ к аккаунту, вход, пароли",
                              "billing": "счета, оплата, возвраты",
                              "shipping": "доставка заказов"}},
        "angry": {"type": "noul", "instructions": "Автор раздражён?"},
    },
}
response = judge(request)
print(json.dumps(response["answers"], ensure_ascii=False, indent=1))
print(response["usage"])

## The four question types

| type | criteria | answer |
|---|---|---|
| `choice` | `{id: description}` | `choice` (id), `probabilities`, `confidence` |
| `score` | `[level 0, level 1, ...]` | `score` (expected level), `probabilities`, `legend` |
| `noul` (yes/no) | optional `{"true": ..., "false": ...}` | `noul` = probability of yes |
| `ranking` | `{id: text}` or `[text, ...]` | `ranking` (ids, best first), `scores`, `probabilities` |

In [ ]:
state = "Сервер с базой заказов не отвечает уже сорок минут, клиенты не могут оформить покупку."
examples = {
    "choice": {"type": "choice", "instructions": "Какой отдел должен этим заняться?",
               "criteria": {"infra": "серверы, сеть, базы данных", "sales": "продажи и клиенты",
                            "hr": "кадры и найм"}},
    "score": {"type": "score", "instructions": "Оцени срочность.",
              "criteria": ["низкая", "средняя", "высокая", "критическая"]},
    "noul": {"type": "noul", "instructions": "Нужна ли срочная эскалация дежурному инженеру?"},
    "ranking": {"type": "ranking", "instructions": "Какое действие поможет первым?",
                "criteria": {"restart": "Перезапустить сервер базы данных и проверить логи.",
                             "email": "Разослать клиентам письмо с извинениями.",
                             "report": "Подготовить квартальный отчёт."}},
}
answers = judge({"state": state, "questions": examples})["answers"]
for qid, answer in answers.items():
    print(qid, "->", {k: v for k, v in answer.items() if k not in ("probabilities", "legend")})

## A big catalog: 243 intents in one request

Packing puts all options of a question into a few rows next to one copy of the state, and the
state cache encodes the state once. Options never attend to each other, so the result is the
same as scoring every option on its own — only cheaper.

In [ ]:
import urllib.request
from pathlib import Path

local = [p for p in (Path("examples/data/intent_catalog.json"),
                     Path("../examples/data/intent_catalog.json")) if p.exists()]
if local:                       # running from a clone of the repository
    catalog = json.loads(local[0].read_text(encoding="utf-8"))
else:
    url = "https://raw.githubusercontent.com/ai-forever/frida-decisions/main/examples/data/intent_catalog.json"
    catalog = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))

counts = judge.token_counts(catalog["request"])
print(f"{counts['candidates']} intents; encoder tokens: one sequence per intent = {counts['naive']}, "
      f"packed = {counts['packed']}, packed + state cache = {counts['cached']}")

for attempt in ("first call", "same state again (cache hit)"):
    started = time.perf_counter()
    response = judge(catalog["request"])
    print(f"{attempt}: {time.perf_counter() - started:.2f}s", response["usage"]["state_cache"])

answer = response["answers"]["intent"]
top = sorted(answer["probabilities"].items(), key=lambda kv: -kv[1])[:5]
print("choice:", answer["choice"], "| expected:", catalog["expected"]["intent"])
print("top-5:", [(k, round(p, 3)) for k, p in top])

## CPU: int8 ONNX

`OnnxJudge` has the same API and needs only `onnxruntime` at inference. The graph is
quantised to int8 with per-token activation scales.

In [ ]:
onnx_judge = OnnxJudge.from_pretrained(MODEL, threads=os.cpu_count())
started = time.perf_counter()
onnx_answers = onnx_judge(request)["answers"]
print(f"{time.perf_counter() - started:.2f}s")
print(json.dumps(onnx_answers, ensure_ascii=False, indent=1))